# Retrieval Evaluation — TF-IDF vs OpenAI Embeddings

This notebook evaluates two retrieval approaches for the AI Diet Coach knowledge base:

| Approach | Method |
|---|---|
| **TF-IDF (minsearch)** | Sparse keyword matching over name, category, area, ingredients, instructions |
| **OpenAI Embeddings** | Dense semantic search using `text-embedding-3-small` + cosine similarity |

We test both on 15 queries spanning exact matches, synonyms, vague descriptions, and cuisine types.
For each query we measure **Hit@3** (did a relevant recipe appear in the top 3 results?).

In [1]:
import json
import time
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
from minsearch import Index

client = OpenAI()

## 1. Load Data

In [2]:
with open('../data/recipes.json') as f:
    western = json.load(f)
with open('../data/asian_recipes.json') as f:
    asian = json.load(f)
    for doc in asian:
        if isinstance(doc.get('ingredients'), list):
            doc['ingredients'] = ', '.join(doc['ingredients'])

documents = western + asian
print(f'Total recipes: {len(documents)}')
print(f'Categories: {sorted({d["category"] for d in documents})[:10]} ...')

Total recipes: 256
Categories: ['Beef', 'Breakfast', 'Chicken', 'Dessert', 'Goat', 'Lamb', 'Miscellaneous', 'Pasta', 'Pork', 'Seafood'] ...


## 2. Build TF-IDF Index

In [3]:
t0 = time.time()
tfidf_index = Index(
    text_fields=['name', 'category', 'area', 'ingredients', 'instructions'],
    keyword_fields=['category', 'area'],
)
tfidf_index.fit(documents)
tfidf_build_time = time.time() - t0
print(f'TF-IDF index built in {tfidf_build_time:.2f}s')

def tfidf_search(query, n=3):
    return tfidf_index.search(query, num_results=n)

TF-IDF index built in 0.02s


## 3. Build Embeddings Index

Each recipe is embedded as: `name | category | area | ingredients (first 100 chars)`

In [4]:
def recipe_to_text(doc):
    ingredients = doc.get('ingredients', '')
    if isinstance(ingredients, list):
        ingredients = ', '.join(ingredients)
    return f"{doc['name']} | {doc.get('category','')} | {doc.get('area','')} | {ingredients[:120]}"

# Embed all recipes in one batch (256 recipes × ~30 tokens ≈ negligible cost)
t0 = time.time()
texts = [recipe_to_text(d) for d in documents]
response = client.embeddings.create(model='text-embedding-3-small', input=texts)
embeddings = np.array([r.embedding for r in response.data])
embed_build_time = time.time() - t0
print(f'Embeddings built in {embed_build_time:.2f}s  shape={embeddings.shape}')

def cosine_search(query, n=3):
    resp = client.embeddings.create(model='text-embedding-3-small', input=[query])
    q_vec = np.array(resp.data[0].embedding)
    scores = embeddings @ q_vec / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(q_vec) + 1e-9)
    top_idx = scores.argsort()[::-1][:n]
    return [documents[i] for i in top_idx]

Embeddings built in 3.61s  shape=(256, 1536)


## 4. Define Test Queries

15 queries covering different retrieval difficulty levels.
Each has a list of `relevant_keywords` — if any appear in the top-3 result names, it counts as a hit.

In [5]:
test_queries = [
    # Exact match queries
    {'query': 'pad thai',              'relevant': ['pad thai'],              'type': 'exact'},
    {'query': 'beef stir fry',         'relevant': ['stir-fry', 'stir fry'],  'type': 'exact'},
    {'query': 'chicken tikka',         'relevant': ['tikka', 'chicken tikka'],'type': 'exact'},

    # Category queries
    {'query': 'seafood dish',          'relevant': ['seafood', 'prawn', 'salmon', 'fish', 'shrimp'], 'type': 'category'},
    {'query': 'vegetarian meal',       'relevant': ['vegetarian', 'tofu', 'lentil', 'aubergine'],    'type': 'category'},
    {'query': 'Korean food',           'relevant': ['korean', 'bibimbap', 'bulgogi', 'kimchi'],      'type': 'category'},
    {'query': 'Thai cuisine',          'relevant': ['thai', 'pad', 'tom yum', 'massaman'],           'type': 'category'},

    # Ingredient queries
    {'query': 'recipe with coconut milk', 'relevant': ['curry', 'laksa', 'rendang', 'thai'],         'type': 'ingredient'},
    {'query': 'garlic and ginger',        'relevant': ['stir', 'asian', 'chicken', 'beef'],          'type': 'ingredient'},

    # Synonym / paraphrase queries (harder for TF-IDF)
    {'query': 'lean protein meal',     'relevant': ['chicken', 'fish', 'salmon', 'prawn'],           'type': 'synonym'},
    {'query': 'healthy bowl',         'relevant': ['bibimbap', 'salad', 'rice bowl'],                'type': 'synonym'},
    {'query': 'noodle soup',          'relevant': ['ramen', 'pho', 'laksa', 'noodle'],               'type': 'synonym'},

    # Vague / goal-based queries
    {'query': 'quick weeknight dinner',       'relevant': ['stir-fry', 'stir fry', 'quick', 'gyudon'], 'type': 'vague'},
    {'query': 'something light and healthy',  'relevant': ['salad', 'fish', 'vegetarian', 'seafood'],   'type': 'vague'},
    {'query': 'comfort food',                 'relevant': ['stew', 'curry', 'congee', 'soup'],          'type': 'vague'},
]

## 5. Run Comparison

In [6]:
def hit_at_3(results, relevant_keywords):
    """True if any top-3 result name contains a relevant keyword."""
    for r in results:
        name_lower = r['name'].lower()
        for kw in relevant_keywords:
            if kw.lower() in name_lower:
                return True
    return False

results_table = []

for tq in test_queries:
    q = tq['query']
    rel = tq['relevant']

    # TF-IDF
    t0 = time.time()
    tfidf_res = tfidf_search(q, n=3)
    tfidf_ms = round((time.time() - t0) * 1000, 1)
    tfidf_hit = hit_at_3(tfidf_res, rel)

    # Embeddings
    t0 = time.time()
    embed_res = cosine_search(q, n=3)
    embed_ms = round((time.time() - t0) * 1000, 1)
    embed_hit = hit_at_3(embed_res, rel)

    results_table.append({
        'query': q,
        'type': tq['type'],
        'tfidf_top3': [r['name'] for r in tfidf_res],
        'tfidf_hit': tfidf_hit,
        'tfidf_ms': tfidf_ms,
        'embed_top3': [r['name'] for r in embed_res],
        'embed_hit': embed_hit,
        'embed_ms': embed_ms,
    })

    hit_icon = lambda h: '✅' if h else '❌'
    print(f"[{tq['type']:<10}] {q}")
    print(f"  TF-IDF  {hit_icon(tfidf_hit)} {[r['name'] for r in tfidf_res]}")
    print(f"  Embeds  {hit_icon(embed_hit)} {[r['name'] for r in embed_res]}")
    print()

[exact     ] pad thai
  TF-IDF  ✅ ['Thai Basil Chicken (Pad Krapow)', 'Pad Thai', 'Pad Thai']
  Embeds  ✅ ['Pad Thai', 'Pad Thai', 'Pad See Ew']



[exact     ] beef stir fry
  TF-IDF  ✅ ['Thai beef stir-fry', 'Beef and Broccoli Stir-Fry', 'Beef Asado']
  Embeds  ✅ ['Thai beef stir-fry', 'Beef and Broccoli Stir-Fry', 'Szechuan Beef']



[exact     ] chicken tikka
  TF-IDF  ❌ ['Chicken Adobo', 'Hainanese Chicken Rice', 'Chicken Ramen']
  Embeds  ❌ ['Tandoori Chicken', 'Indian Butter Chicken (Murgh Makhani)', 'Red curry chicken kebabs']

[category  ] seafood dish
  TF-IDF  ✅ ['Arroz con gambas y calamar', 'Korean Sundubu Jjigae (Soft Tofu Stew)', 'Bang bang prawn salad']
  Embeds  ✅ ['Chinese Steamed Fish', 'Sea bass with sizzled ginger, chilli & spring onions', 'Shrimp With Snow Peas']



[category  ] vegetarian meal
  TF-IDF  ❌ ['Air fryer patatas bravas', 'Palak Paneer', 'Dal Tadka']
  Embeds  ❌ ['Vietnamese veg parcels', 'Vegan Lasagna', 'Gado-Gado']

[category  ] Korean food
  TF-IDF  ✅ ['Korean Kimchi Jjigae', 'Japchae (Korean Glass Noodles)', 'Korean Sundubu Jjigae (Soft Tofu Stew)']
  Embeds  ✅ ['Bulgogi', 'Korean Kimchi Jjigae', 'Bibimbap']



[category  ] Thai cuisine
  TF-IDF  ✅ ['Thai Green Curry', 'Thai Basil Chicken (Pad Krapow)', 'Thai Green Curry']
  Embeds  ✅ ['Pad Thai', 'Pad Thai', 'Thai Green Curry']

[ingredient] recipe with coconut milk
  TF-IDF  ✅ ['Grilled eggplant with coconut milk', 'Thai coconut & veg broth', 'Thai Green Curry']
  Embeds  ✅ ['Thai coconut & veg broth', 'Seri muka kuih', 'Nasi lemak']



[ingredient] garlic and ginger
  TF-IDF  ❌ ['French Lentils With Garlic and Thyme', 'Vietnamese Caramelised Ginger Fish (Ca Kho To)', 'Chilli ginger lamb chops']
  Embeds  ✅ ['Chilli ginger lamb chops', 'Vietnamese-style veggie hotpot', 'Lemongrass beef stew with noodles']

[synonym   ] lean protein meal
  TF-IDF  ❌ ['Aussie Burgers', 'Lasagna Sandwiches']
  Embeds  ❌ ['Vegan Lasagna', 'Beef Lo Mein', 'Beef and Broccoli Stir-Fry']



[synonym   ] healthy bowl
  TF-IDF  ✅ ['Noodle bowl salad', 'Gyudon (Japanese Beef Bowl)', 'Japanese Oyakodon (Chicken and Egg Bowl)']
  Embeds  ✅ ['Gyudon (Japanese Beef Bowl)', 'Beef Banh Mi Bowls with Sriracha Mayo, Carrot & Pickled Cucumber', 'Noodle bowl salad']

[synonym   ] noodle soup
  TF-IDF  ✅ ['Salmon noodle soup', 'Thai curry noodle soup', 'Steak & Vietnamese noodle salad']
  Embeds  ✅ ['Salmon noodle soup', 'Ramen Noodles with Boiled Egg', 'Thai curry noodle soup']



[vague     ] quick weeknight dinner
  TF-IDF  ✅ ['Quick gazpacho', 'Salmon noodle wraps', 'Cajun spiced fish tacos']
  Embeds  ✅ ['Drunken noodles (pad kee mao)', 'Beef and Broccoli Stir-Fry', 'Chicken Alfredo Primavera']

[vague     ] something light and healthy
  TF-IDF  ❌ ['Sweet and Sour Pork', 'Sweet and Sour Chicken', 'Hot and Sour Soup']
  Embeds  ✅ ['Fasoliyyeh Bi Z-Zayt (Syrian Green Beans with Olive Oil)', 'Som Tam (Papaya Salad)', 'Quick gazpacho']



[vague     ] comfort food
  TF-IDF  ✅ ['Thai chicken cakes with sweet chilli sauce', 'Thai drumsticks', 'Lemongrass beef stew with noodles']
  Embeds  ✅ ['Filipino Arroz Caldo (Rice Congee)', 'Chinese Congee (Jook)', 'Pancit Canton']



## 6. Metrics Summary

In [7]:
import pandas as pd

df = pd.DataFrame(results_table)

# Overall Hit@3
tfidf_hit_rate = df['tfidf_hit'].mean()
embed_hit_rate = df['embed_hit'].mean()

# Hit@3 by query type
by_type = df.groupby('type')[['tfidf_hit', 'embed_hit']].mean()

# Latency
tfidf_avg_ms = df['tfidf_ms'].mean()
embed_avg_ms = df['embed_ms'].mean()

print('=' * 55)
print(f'  Overall Hit@3   TF-IDF: {tfidf_hit_rate:.0%}   Embeddings: {embed_hit_rate:.0%}')
print(f'  Avg latency     TF-IDF: {tfidf_avg_ms:.1f}ms   Embeddings: {embed_avg_ms:.0f}ms')
print('=' * 55)
print()
print('Hit@3 by query type:')
print(by_type.rename(columns={'tfidf_hit': 'TF-IDF', 'embed_hit': 'Embeddings'}).to_string())

# Wins and losses
tfidf_only = df[df['tfidf_hit'] & ~df['embed_hit']]
embed_only = df[~df['tfidf_hit'] & df['embed_hit']]
print(f'\nTF-IDF wins (hit where embeddings missed): {len(tfidf_only)}')
for _, row in tfidf_only.iterrows():
    print(f"  '{row['query']}'")
print(f'Embeddings wins (hit where TF-IDF missed): {len(embed_only)}')
for _, row in embed_only.iterrows():
    print(f"  '{row['query']}'")

  Overall Hit@3   TF-IDF: 67%   Embeddings: 80%
  Avg latency     TF-IDF: 4.8ms   Embeddings: 200ms

Hit@3 by query type:
              TF-IDF  Embeddings
type                            
category    0.750000    0.750000
exact       0.666667    0.666667
ingredient  0.500000    1.000000
synonym     0.666667    0.666667
vague       0.666667    1.000000

TF-IDF wins (hit where embeddings missed): 0
Embeddings wins (hit where TF-IDF missed): 2
  'garlic and ginger'
  'something light and healthy'


## 7. Cost and Latency Comparison

In [8]:
# text-embedding-3-small pricing: $0.02 per 1M tokens
# Avg recipe text ~40 tokens, query ~8 tokens
n_recipes = len(documents)
index_tokens = n_recipes * 40
query_tokens = 8
price_per_M = 0.02

index_cost = index_tokens / 1_000_000 * price_per_M
query_cost = query_tokens / 1_000_000 * price_per_M

print('Cost analysis (text-embedding-3-small at $0.02/1M tokens):')
print(f'  Index build ({n_recipes} recipes × 40 tokens): ${index_cost:.5f}')
print(f'  Per query   (8 tokens):                        ${query_cost:.7f}')
print(f'  10,000 queries/day:                            ${query_cost * 10000:.3f}/day')
print()
print('Latency:')
print(f'  TF-IDF:     {tfidf_avg_ms:.1f}ms avg  (pure in-memory, no network)')
print(f'  Embeddings: {embed_avg_ms:.0f}ms avg  (includes OpenAI API round-trip)')

Cost analysis (text-embedding-3-small at $0.02/1M tokens):
  Index build (256 recipes × 40 tokens): $0.00020
  Per query   (8 tokens):                        $0.0000002
  10,000 queries/day:                            $0.002/day

Latency:
  TF-IDF:     4.8ms avg  (pure in-memory, no network)
  Embeddings: 200ms avg  (includes OpenAI API round-trip)


## 8. Conclusion

Based on the evaluation above:

| Criterion | TF-IDF (minsearch) | OpenAI Embeddings |
|---|---|---|
| **Hit@3 (overall)** | _(see results)_ | _(see results)_ |
| **Hit@3 (exact match)** | Strong | Strong |
| **Hit@3 (synonym/vague)** | Weaker | Stronger |
| **Index build time** | <0.1s | ~5-10s |
| **Query latency** | <5ms | ~200-400ms |
| **Cost** | Free | ~$0 for this scale |
| **External dependency** | None | OpenAI API |

**Decision: TF-IDF (minsearch) was chosen for this project because:**

1. Recipe search is keyword-heavy — users say "chicken", "Thai", "pasta" which are exact terms in the index
2. Zero latency and zero cost per query matters when every agent turn triggers retrieval
3. Hit@3 on the exact-match and category queries (the dominant use cases) is competitive with embeddings
4. No external API dependency makes the search robust (embeddings would fail if OpenAI is down)
5. The semantic gap is partially mitigated by the agent tools — users can clarify vague requests in follow-up turns

Embeddings would be preferred if the query mix had more synonym/vague queries and latency/cost were not constraints.